# 08 Pitch Choice Diagnostics Source

Converted from `code/pitch_choice_diagnostics.py` for Colab/code review. Edit/comment cells here, then save the reviewed notebook.


Run this notebook from the repository root after cloning the project in Colab. The production script remains in `code/`; this notebook is a review/editing copy.

In [ ]:
from pathlib import Path ## Import Path for object-oriented filesystem paths
import os ## Import os for interacting with the operating system

## Check if 'requirements.txt' exists to infer if the current directory is the repository root
if Path('requirements.txt').exists():
    print('Repository root detected:', Path.cwd()) ## Inform user that the repository root is detected
else:
    print('Clone/cd into the repository root before running script cells.') ## Prompt user to navigate to the repository root

In [ ]:
#!/usr/bin/env python3
"""Diagnose and tune next-pitch prediction models.""" ## Module purpose: Diagnose and tune next-pitch prediction models

from __future__ import annotations ## Enable postponed evaluation of type annotations

import json ## Import json for working with JSON data
import os ## Import os for interacting with the operating system, e.g., environment variables
import argparse ## Import argparse for parsing command-line arguments
from pathlib import Path ## Import Path for object-oriented filesystem paths
from typing import Any ## Import Any for flexible type hinting (used when type is unknown or dynamic)

## Set environment variables for cache directories to centralize cached files
os.environ.setdefault("MPLCONFIGDIR", str(Path(".cache/matplotlib").resolve())) ## Matplotlib cache directory
os.environ.setdefault("PYBASEBALL_CACHE", str(Path(".cache/pybaseball").resolve())) ## PyBaseball cache directory
os.environ.setdefault("XDG_CACHE_HOME", str(Path(".cache").resolve())) ## XDG cache home directory

import matplotlib ## Import matplotlib, a comprehensive library for creating static, animated, and interactive visualizations in Python
matplotlib.use("Agg") ## Use the 'Agg' backend for Matplotlib, which is suitable for non-interactive plotting (e.g., saving to file) and ideal for server-side use where a display is not available

import matplotlib.pyplot as plt ## Import pyplot for creating plots and customizing them
import numpy as np ## Import numpy for high-performance numerical operations, especially array manipulation
import pandas as pd ## Import pandas for data manipulation and analysis, primarily with DataFrames
import seaborn as sns ## Import seaborn for statistical data visualization, built on matplotlib
from sklearn.compose import ColumnTransformer ## Import ColumnTransformer for applying different transformations to different columns of an array or DataFrame
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier ## Import ensemble models for classification tasks
from sklearn.impute import SimpleImputer ## Import SimpleImputer for handling missing values in datasets
from sklearn.linear_model import LogisticRegression ## Import LogisticRegression, a linear model for classification
from sklearn.metrics import accuracy_score, f1_score ## Import metrics for evaluating classification model performance
from sklearn.model_selection import train_test_split ## Import train_test_split for splitting datasets into training and testing subsets
from sklearn.pipeline import Pipeline ## Import Pipeline for chaining multiple processing steps into a single scikit-learn estimator
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler ## Import preprocessors for feature engineering: OneHotEncoder for categorical, OrdinalEncoder for ordinal, StandardScaler for numerical scaling

from pitch_sequence_pipeline import PROCESSED_DIR, ROOT, log, pitch_label, sequence_features_path, temporal_sequence_features_path ## Import custom utilities specific to the pitch sequence pipeline

## Define output and figure directories relative to the project root
OUTPUT_DIR = ROOT / "output" ## Path to the main output directory
FIGURE_DIR = OUTPUT_DIR / "figures" ## Path to the directory for saving figures/plots

NEXT_CAT = [ ## List of categorical features used for next-pitch prediction models
    "pitcher_name", ## Name of the pitcher
    "batter_name", ## Name of the batter
    "stand", ## Batter's handedness (left/right)
    "p_throws", ## Pitcher's handedness (left/right)
    "count", ## Ball-strike count in a canonical string format (e.g., '0-0', '1-2')
    "base_state", ## State of runners on base
    "prev_pitch_type", ## Type of the previous pitch
    "prev2_pitch_type", ## Type of the pitch before the previous one
    "prev_description", ## Description of the previous pitch outcome
    "inning_topbot", ## Whether it's the top or bottom of the inning
]

NEXT_NUM = [ #  numerical features used for next-pitch prediction models, ex number of ballsm number of strikes
    "balls", ## Number of balls in the current count
    "strikes", ## Number of strikes in the current count
    "outs_when_up", ## Number of outs when the batter came up
    "inning", ## Current inning number
    "pitch_number", ## Sequential number of the pitch in the at-bat
    "prev_zone", ## Zone of the previous pitch (from 1-14)
    "prev_release_speed", ## Speed of the previous pitch at release
    "prev_plate_x", ## Horizontal location of the previous pitch as it crossed the plate
    "prev_plate_z", ## Vertical location of the previous pitch as it crossed the plate
    "score_diff_batting_team", ## Difference in score for the batting team
    "lineup_left_share",
    "lineup_right_share",
    "lineup_switch_share", ## Share of switch-hitting batters in the current lineup
    "lineup_batter_count", #
]

In [ ]:
def ensure_dirs() -> None: ## Function to ensure that output directories exist
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True) ## Create the main output directory if it doesn't exist, including any necessary parent directories
    FIGURE_DIR.mkdir(parents=True, exist_ok=True) #

In [ ]:
def save_fig(path: Path) -> None: ## Function to save a matplotlib figure to a specified path
    plt.tight_layout() ## Adjust plot parameters for a tight layout, preventing labels from overlapping
    plt.savefig(path, dpi=220, bbox_inches="tight") ## Save the current figure to the given path with high DPI and tight bounding box
    plt.close() ## Close the current figure to free up memory
    log(f"wrote {path.relative_to(ROOT)}") ## Log the path of the saved figure relative to the project root

In [ ]:
def load_features(top_n: int) -> tuple[pd.DataFrame, pd.DataFrame]: ## Function to load pitch sequence features for specific years
    ## Load 2025 features from a parquet file. 'sequence_features_path' constructs the file path.
    df_2025 = pd.read_parquet(sequence_features_path(2025, top_n))

    ## Construct the path for 2026 temporal sequence features.
    path_2026 = temporal_sequence_features_path(2025, 2026, top_n)

    ## Load 2026 features if the file exists, otherwise create an empty DataFrame.
    ## This handles cases where 2026 data might not yet be available.
    df_2026 = pd.read_parquet(path_2026) if path_2026.exists() else pd.DataFrame()

    ## Return both DataFrames: 2025 features and 2026 features.
    return df_2025, df_2026

In [ ]:
def model_frame(
    df: pd.DataFrame, ## Input DataFrame containing raw pitch data
    keep_pitch_types: pd.Index | None = None, ## Optional: A list of pitch types to keep. If None, it's determined from data.
    min_pitch_type_count: int = 1, ## Minimum occurrences for a pitch type to be considered for 'keep_pitch_types'
) -> pd.DataFrame: ## Returns a filtered DataFrame ready for modeling
    out = df.copy() ## Create a copy to avoid modifying the original DataFrame
    if out.empty:
        return out ## If the DataFrame is empty, return it as is
    if keep_pitch_types is None: ## If specific pitch types to keep are not provided
        counts = out["pitch_type"].value_counts() ## Calculate frequency of each pitch type
        keep_pitch_types = counts.loc[counts >= min_pitch_type_count].index ## Filter pitch types by minimum count
    return out.loc[out["pitch_type"].isin(keep_pitch_types)].copy() ## Return DataFrame with only the specified pitch types

In [ ]:
def feature_columns(df: pd.DataFrame) -> tuple[list[str], list[str], list[str]]: ## Function to identify categorical and numerical features present in the DataFrame
    cat = [col for col in NEXT_CAT if col in df.columns] ## Filter NEXT_CAT for columns actually present in df
    num = [col for col in NEXT_NUM if col in df.columns] ## Filter NEXT_NUM for columns actually present in df
    return cat, num, cat + num ## Return lists of categorical features, numerical features, and all features combined

In [ ]:
def top_k_accuracy_from_probs(classes: np.ndarray, probs: np.ndarray, y: pd.Series, k: int) -> float:
    ## Calculate the order of predictions by probability in descending order
    order = np.argsort(probs, axis=1)[:, -min(k, len(classes)) :] ## Select top 'k' indices based on probabilities
    top = classes[order] ## Get the actual class names for the top 'k' predicted indices
    ## Check if the true label is among the top 'k' predictions for each instance and return the mean accuracy
    return float(np.mean([actual in choices for actual, choices in zip(y.to_numpy(), top)]))

In [ ]:
def evaluate_probs(name: str, dataset: str, classes: np.ndarray, probs: np.ndarray, y: pd.Series) -> dict[str, Any]:
    pred = classes[np.argmax(probs, axis=1)] ## Get the class with the highest probability for each prediction
    class_to_idx = {pitch: idx for idx, pitch in enumerate(classes)} ## Map class names to their indices
    ## Get the probability of the actual class for each instance
    actual_prob = np.array([probs[i, class_to_idx[pitch]] if pitch in class_to_idx else 0.0 for i, pitch in enumerate(y)])
    return {
        "model": name, ## Name of the model being evaluated
        "dataset": dataset, ## Name of the dataset (e.g., 'training', 'holdout')
        "rows": int(len(y)), ## Number of samples in the evaluation dataset
        "accuracy": float(accuracy_score(y, pred)), ## Calculate standard accuracy (top-1 accuracy)
        "top2_accuracy": top_k_accuracy_from_probs(classes, probs, y, 2), ## Calculate top-2 accuracy
        "top3_accuracy": top_k_accuracy_from_probs(classes, probs, y, 3), ## Calculate top-3 accuracy
        "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0)), ## Calculate macro-averaged F1-score
        "avg_actual_probability": float(actual_prob.mean()), ## Average probability assigned to the actual class
        "avg_confidence": float(probs.max(axis=1).mean()), ## Average confidence (max probability) of the model's predictions
    }

In [ ]:
def build_pitcher_arsenal_table(
    rows: pd.DataFrame, ## Input DataFrame containing pitch data
    y: pd.Series, ## Series containing the target pitch types for each row in 'rows'
    min_usage: float = 0.03, ## Minimum usage rate for a pitch to be considered part of a pitcher's arsenal
    min_pitches: int = 30, ## Minimum number of pitches for a pitch to be considered part of a pitcher's arsenal
) -> pd.DataFrame: ## Returns a DataFrame detailing pitcher arsenals
    data = rows.loc[y.index].copy() ## Filter input rows to match the target series' index
    data["_target_pitch"] = y.to_numpy() ## Add target pitch as a new column

    agg_spec: dict[str, tuple[str, str]] = { ## Aggregation specifications for groupby operation
        "pitches": ("_target_pitch", "size"), ## Count of each pitch type
    }
    optional_means = { ## Dictionary of optional columns to calculate their mean
        "avg_release_speed": "release_speed",
        "avg_release_spin_rate": "release_spin_rate",
        "whiff_rate": "whiff",
        "weak_contact_rate": "weak_contact",
        "whiff_or_weak_rate": "whiff_or_weak",
        "mean_pitcher_run_value": "pitcher_run_value",
    }
    for output_col, source_col in optional_means.items():
        if source_col in data.columns: ## Only add if source column exists in data
            agg_spec[output_col] = (source_col, "mean") ## Add mean aggregation for existing columns

    arsenal = (
        data.groupby(["pitcher_name", "_target_pitch"], as_index=False) ## Group by pitcher and target pitch
        .agg(**agg_spec) ## Apply aggregations
        .rename(columns={"_target_pitch": "pitch_type"}) ## Rename target pitch column to 'pitch_type'
    )
    arsenal["pitch_label"] = arsenal["pitch_type"].map(pitch_label) ## Map pitch types to human-readable labels
    arsenal["pitcher_total_pitches"] = arsenal.groupby("pitcher_name")["pitches"].transform("sum") ## Calculate total pitches for each pitcher
    arsenal["usage_rate"] = arsenal["pitches"] / arsenal["pitcher_total_pitches"] ## Calculate usage rate for each pitch type
    ## Determine if a pitch is in a pitcher's arsenal based on usage and total pitch count
    arsenal["in_arsenal"] = (arsenal["usage_rate"] >= min_usage) | (arsenal["pitches"] >= min_pitches)

    ## Guarantee every pitcher has at least one allowed pitch if thresholds are too strict.
    ## Select the top pitch by count for each pitcher
    top_idx = arsenal.sort_values(["pitcher_name", "pitches"], ascending=[True, False]).groupby("pitcher_name").head(1).index
    arsenal.loc[top_idx, "in_arsenal"] = True ## Mark the top pitch as in arsenal

    arsenal["arsenal_size"] = arsenal.groupby("pitcher_name")["in_arsenal"].transform("sum").astype(int) ## Calculate arsenal size for each pitcher
    arsenal = arsenal.sort_values(["pitcher_name", "in_arsenal", "usage_rate"], ascending=[True, False, False]) ## Sort the arsenal table
    arsenal.to_csv(OUTPUT_DIR / "pitcher_arsenals_2025_training.csv", index=False) ## Save the arsenal table to CSV
    return arsenal ## Return the pitcher arsenal DataFrame

In [ ]:
def arsenal_map_from_table(arsenal: pd.DataFrame) -> dict[str, set[str]]: ## Function to create a mapping from pitcher name to their allowed pitch types
    allowed = arsenal.loc[arsenal["in_arsenal"]].copy() ## Filter for pitches that are in a pitcher's arsenal
    ## Group by pitcher name and collect all allowed pitch types into a set for each pitcher
    return allowed.groupby("pitcher_name")["pitch_type"].apply(lambda s: set(s.astype(str))).to_dict()

In [ ]:
def apply_arsenal_mask(
    probs: np.ndarray, ## Predicted probabilities from the model
    pitcher_names: pd.Series, ## Series of pitcher names corresponding to the probabilities
    classes: np.ndarray, ## Array of all possible pitch classes
    arsenal_map: dict[str, set[str]], ## Map from pitcher name to their allowed pitch types
) -> np.ndarray: ## Returns masked probabilities where disallowed pitches have zero probability
    class_index = {pitch: idx for idx, pitch in enumerate(classes)} ## Map pitch class names to their column index in probs array
    masked = np.zeros_like(probs) ## Initialize an array of zeros with the same shape as probs

    for i, pitcher_name in enumerate(pitcher_names.astype(str).to_numpy()): ## Iterate through each prediction and corresponding pitcher
        allowed = arsenal_map.get(pitcher_name) ## Get the set of allowed pitches for the current pitcher
        if not allowed: ## If no allowed pitches are defined for this pitcher (e.g., new pitcher)
            masked[i] = probs[i] ## Keep original probabilities for this pitcher
            continue

        ## Get indices of allowed pitches that are also present in the model's classes
        allowed_idx = [class_index[pitch] for pitch in allowed if pitch in class_index]
        if not allowed_idx: ## If no allowed pitches are found in the model's classes
            masked[i] = probs[i] ## Keep original probabilities for this pitcher
            continue

        row = np.zeros(probs.shape[1], dtype=float) ## Create a zero-filled row for the masked probabilities
        row[allowed_idx] = probs[i, allowed_idx] ## Copy probabilities for allowed pitches
        total = row.sum() ## Calculate the sum of probabilities for allowed pitches
        if total > 0:
            row = row / total ## Normalize probabilities if sum is greater than zero
        else:
            row[allowed_idx] = 1 / len(allowed_idx) ## If sum is zero, assign equal probability to allowed pitches
        masked[i] = row ## Assign the processed row to the masked probabilities

    return masked ## Return the array of masked probabilities

In [ ]:
def distribution_lookup_predict(
    train: pd.DataFrame, ## Training DataFrame used to learn pitch distributions
    target: pd.Series, ## Target variable (pitch type) for the training data
    test: pd.DataFrame, ## Test DataFrame for which predictions are to be made
    classes: np.ndarray, ## Array of all possible pitch classes
    keys: list[str], ## Primary keys to group by for calculating distributions
    fallback_keys: list[list[str]], ## List of fallback key sets to use if primary keys don't yield a distribution
    alpha: float = 0.0, ## Laplace smoothing parameter (alpha) to handle zero counts
) -> np.ndarray: ## Returns an array of predicted probabilities for each test instance
    class_index = {pitch: i for i, pitch in enumerate(classes)} ## Map class names to their index in the output array
    global_counts = target.value_counts().reindex(classes, fill_value=0).astype(float) ## Global counts of each pitch type in training data
    global_probs = (global_counts + alpha) / (global_counts.sum() + alpha * len(classes)) ## Global probabilities with Laplace smoothing

    tables = [] ## List to store lookup tables (key sets and their probability distributions)
    train_with_y = train.copy() ## Create a copy of training data
    train_with_y["_target_pitch"] = target.to_numpy() ## Add target pitch column to training data
    for key_set in [keys] + fallback_keys: ## Iterate through primary keys and fallback keys
        if not key_set:
            continue ## Skip if key set is empty
        counts = train_with_y.groupby(key_set + ["_target_pitch"]).size().unstack(fill_value=0) ## Group by keys and target, then unstack to get counts
        counts = counts.reindex(columns=classes, fill_value=0).astype(float) ## Reindex columns to ensure all classes are present, fill missing with 0
        probs = (counts + alpha).div(counts.sum(axis=1) + alpha * len(classes), axis=0) ## Calculate probabilities with Laplace smoothing
        tables.append((key_set, probs)) ## Add the key set and its probability table to the list

    out = np.zeros((len(test), len(classes)), dtype=float) ## Initialize output array for probabilities
    for i, row in enumerate(test.itertuples(index=False)): ## Iterate through each row in the test data
        row_map = row._asdict() ## Convert row to a dictionary for easy key access
        chosen = None ## Initialize chosen distribution to None
        for key_set, table in tables: ## Iterate through lookup tables from most specific to least specific
            key = tuple(row_map[col] for col in key_set) ## Construct the key for the current row and key set
            if len(key) == 1:
                key = key[0] ## If key has only one element, extract it
            if key in table.index: ## If the key is found in the current lookup table
                chosen = table.loc[key].to_numpy(dtype=float) ## Use the distribution from this table
                break ## Break since a matching distribution is found
        if chosen is None: ## If no specific distribution is found after checking all tables
            chosen = global_probs.to_numpy(dtype=float) ## Fallback to global probabilities
        out[i] = chosen ## Assign the chosen probability distribution to the output array
    return out ## Return the array of predicted probabilities

In [ ]:
def make_logistic(cat: list[str], num: list[str], class_weight: str | None, c_value: float = 1.0) -> Pipeline: ## Function to create a scikit-learn pipeline for Logistic Regression
    cat_pipe = Pipeline(
        [
            ("impute", SimpleImputer(strategy="constant", fill_value="missing")), ## Impute missing categorical values with 'missing'
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5)), ## One-hot encode categorical features, ignoring unknown categories and categories with less than 5 occurrences
        ]
    ) ## Pipeline for categorical features
    num_pipe = Pipeline(
        [
            ("impute", SimpleImputer(strategy="median")), ## Impute missing numerical values with the median
            ("scale", StandardScaler(with_mean=False)), ## Scale numerical features using StandardScaler without centering
        ]
    ) ## Pipeline for numerical features
    pre = ColumnTransformer([("cat", cat_pipe, cat), ("num", num_pipe, num)]) ## Combine categorical and numerical pipelines using ColumnTransformer
    return Pipeline(
        [
            ("preprocess", pre), ## Preprocessing step using the ColumnTransformer
            (
                "model",
                LogisticRegression(
                    C=c_value, ## Inverse of regularization strength
                    max_iter=1800, ## Maximum number of iterations for the solver
                    solver="lbfgs", ## Algorithm to use in the optimization problem
                    class_weight=class_weight, ## Weights associated with classes (e.g., 'balanced')
                ),
            ),
        ]
    ) ## Final pipeline combining preprocessing and Logistic Regression model

In [ ]:
def make_hgb(cat: list[str], num: list[str]) -> Pipeline:
    cat_pipe = Pipeline(
        [
            ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
            ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ]
    )
    num_pipe = Pipeline([("impute", SimpleImputer(strategy="median"))])
    pre = ColumnTransformer([("cat", cat_pipe, cat), ("num", num_pipe, num)])
    return Pipeline(
        [
            ("preprocess", pre),
            (
                "model",
                HistGradientBoostingClassifier(
                    learning_rate=0.05,
                    max_iter=240,
                    max_leaf_nodes=31,
                    l2_regularization=0.02,
                    random_state=42,
                ),
            ),
        ]
    )


In [ ]:
def make_random_forest(cat: list[str], num: list[str]) -> Pipeline:
    cat_pipe = Pipeline(
        [
            ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
            ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ]
    )
    num_pipe = Pipeline([("impute", SimpleImputer(strategy="median"))])
    pre = ColumnTransformer([("cat", cat_pipe, cat), ("num", num_pipe, num)])
    return Pipeline(
        [
            ("preprocess", pre),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=160,
                    min_samples_leaf=12,
                    max_features="sqrt",
                    n_jobs=-1,
                    random_state=42,
                ),
            ),
        ]
    )


In [ ]:
def model_probs(model: Pipeline, X: pd.DataFrame, classes: np.ndarray) -> np.ndarray:
    raw = model.predict_proba(X)
    out = np.zeros((len(X), len(classes)), dtype=float)
    class_to_idx = {pitch: i for i, pitch in enumerate(model.classes_)}
    for j, pitch in enumerate(classes):
        if pitch in class_to_idx:
            out[:, j] = raw[:, class_to_idx[pitch]]
    row_sum = out.sum(axis=1)
    missing = row_sum == 0
    out[missing, :] = 1 / len(classes)
    out[~missing, :] = out[~missing, :] / row_sum[~missing, None]
    return out


In [ ]:
def audit_best_model(
    model_name: str,
    model: Pipeline,
    classes: np.ndarray,
    X: pd.DataFrame,
    y: pd.Series,
    rows: pd.DataFrame,
    dataset: str,
    arsenal_map: dict[str, set[str]] | None = None,
) -> pd.DataFrame:
    probs = model_probs(model, X, classes)
    if arsenal_map is not None:
        probs = apply_arsenal_mask(probs, X["pitcher_name"], classes, arsenal_map)
    order = np.argsort(probs, axis=1)[:, ::-1]
    pred = classes[order[:, 0]]
    class_to_idx = {pitch: idx for idx, pitch in enumerate(classes)}
    actual_idx = np.array([class_to_idx[pitch] for pitch in y])

    out = rows.loc[X.index, ["game_date", "pitcher_name", "batter_name", "count", "prev_pitch_type", "prev_description"]].copy()
    out.insert(0, "dataset", dataset)
    out.insert(0, "model", model_name)
    out["actual_pitch"] = y.to_numpy()
    out["predicted_pitch"] = pred
    out["actual_pitch_label"] = out["actual_pitch"].map(pitch_label)
    out["predicted_pitch_label"] = out["predicted_pitch"].map(pitch_label)
    out["correct_top1"] = out["actual_pitch"].eq(out["predicted_pitch"])
    out["correct_top2"] = [actual_idx[i] in order[i, :2] for i in range(len(y))]
    out["correct_top3"] = [actual_idx[i] in order[i, :3] for i in range(len(y))]
    out["actual_pitch_probability"] = probs[np.arange(len(y)), actual_idx]
    out["model_confidence"] = probs.max(axis=1)
    out["top_3_predictions"] = [
        ", ".join(f"{pitch_label(classes[j])} {probs[i, j]:.1%}" for j in order[i, :3])
        for i in range(len(y))
    ]
    return out


In [ ]:
def plot_pitcher_arsenals(arsenal: pd.DataFrame) -> None:
    allowed = arsenal.loc[arsenal["in_arsenal"]].copy()
    allowed.to_csv(OUTPUT_DIR / "pitcher_arsenals_2025_training_allowed_only.csv", index=False)
    pivot = allowed.pivot_table(
        index="pitcher_name",
        columns="pitch_type",
        values="usage_rate",
        fill_value=0,
    )
    order = allowed.groupby("pitcher_name")["arsenal_size"].max().sort_values(ascending=True).index
    pivot = pivot.reindex(order)

    ax = pivot.plot(kind="barh", stacked=True, figsize=(10, 8), width=0.82)
    ax.set_title("Pitcher-Specific Arsenals From 2025 Training Data")
    ax.set_xlabel("Pitch usage share")
    ax.set_ylabel("")
    ax.legend(title="Pitch", bbox_to_anchor=(1.02, 1), loc="upper left")
    save_fig(FIGURE_DIR / "pitcher_specific_arsenals_2025_training.png")


In [ ]:
def add_unmasked_and_arsenal_metrics(
    metrics: list[dict[str, Any]],
    name: str,
    dataset: str,
    classes: np.ndarray,
    probs: np.ndarray,
    y: pd.Series,
    pitcher_names: pd.Series,
    arsenal_map: dict[str, set[str]],
) -> None:
    metrics.append(evaluate_probs(name, dataset, classes, probs, y))
    masked = apply_arsenal_mask(probs, pitcher_names, classes, arsenal_map)
    metrics.append(evaluate_probs(f"{name}_arsenal_masked", dataset, classes, masked, y))


In [ ]:
def plot_sweep(metrics: pd.DataFrame) -> None:
    subset = metrics.loc[metrics["dataset"].isin(["2025_holdout", "2026_to_date"])].copy()
    subset = subset.loc[~subset["model"].str.contains("dummy")]
    order = (
        subset.loc[subset["dataset"].eq("2025_holdout")]
        .sort_values("accuracy", ascending=False)["model"]
        .tolist()
    )
    subset["model"] = pd.Categorical(subset["model"], categories=order, ordered=True)

    plt.figure(figsize=(11, 6))
    sns.barplot(data=subset, x="accuracy", y="model", hue="dataset")
    plt.title("Next-Pitch Exact-Match Accuracy: Model Sweep")
    plt.xlabel("Accuracy")
    plt.ylabel("")
    save_fig(FIGURE_DIR / "next_pitch_model_sweep_accuracy.png")

    plt.figure(figsize=(11, 6))
    sns.barplot(data=subset, x="top3_accuracy", y="model", hue="dataset")
    plt.title("Next-Pitch Top-3 Accuracy: Model Sweep")
    plt.xlabel("Top-3 accuracy")
    plt.ylabel("")
    plt.xlim(0, 1)
    save_fig(FIGURE_DIR / "next_pitch_model_sweep_top3.png")


In [ ]:
def plot_mix_for_best(audit: pd.DataFrame, name: str) -> None:
    rows = []
    for dataset, sub in audit.groupby("dataset"):
        for col, label in [("actual_pitch", "Actual"), ("predicted_pitch", "Predicted")]:
            share = sub[col].value_counts(normalize=True)
            rows.extend(
                {
                    "dataset": dataset,
                    "mix_type": label,
                    "pitch_type": pitch,
                    "pitch_label": pitch_label(pitch),
                    "share": value,
                }
                for pitch, value in share.items()
            )
    mix = pd.DataFrame(rows)
    mix.to_csv(OUTPUT_DIR / f"{name}_actual_vs_predicted_pitch_mix.csv", index=False)

    datasets = mix["dataset"].unique().tolist()
    fig, axes = plt.subplots(len(datasets), 1, figsize=(10, 4.5 * len(datasets)), sharey=True)
    if len(datasets) == 1:
        axes = [axes]
    for ax, dataset in zip(axes, datasets):
        sub = mix.loc[mix["dataset"].eq(dataset)]
        order = sub.loc[sub["mix_type"].eq("Actual")].sort_values("share", ascending=False)["pitch_label"]
        sns.barplot(data=sub, x="pitch_label", y="share", hue="mix_type", order=order, ax=ax)
        ax.set_title(f"Actual vs Predicted Pitch Mix: {dataset} ({name})")
        ax.set_xlabel("")
        ax.set_ylabel("Share")
        ax.tick_params(axis="x", rotation=35)
    save_fig(FIGURE_DIR / f"{name}_actual_vs_predicted_pitch_mix.png")


In [ ]:
def context_entropy(train: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    df = train.copy()
    df["_target_pitch"] = y.to_numpy()
    rows = []
    for label, keys in {
        "pitcher": ["pitcher_name"],
        "pitcher_count": ["pitcher_name", "count"],
        "pitcher_prev": ["pitcher_name", "prev_pitch_type"],
        "pitcher_count_prev": ["pitcher_name", "count", "prev_pitch_type"],
        "pitcher_count_prev_stand": ["pitcher_name", "count", "prev_pitch_type", "stand"],
    }.items():
        counts = df.groupby(keys + ["_target_pitch"]).size().unstack(fill_value=0)
        probs = counts.div(counts.sum(axis=1), axis=0)
        max_share = probs.max(axis=1)
        entropy = -(probs.where(probs > 0, 1).map(np.log2) * probs).sum(axis=1)
        weights = counts.sum(axis=1) / counts.sum(axis=1).sum()
        rows.append(
            {
                "context": label,
                "groups": int(len(counts)),
                "weighted_mode_share": float((max_share * weights).sum()),
                "weighted_entropy_bits": float((entropy * weights).sum()),
                "median_group_pitches": float(counts.sum(axis=1).median()),
            }
        )
    out = pd.DataFrame(rows)
    out.to_csv(OUTPUT_DIR / "pitch_choice_context_entropy.csv", index=False)
    return out


In [ ]:
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Run next-pitch model sweep and context diagnostics.")
    parser.add_argument("--top-n", type=int, default=100)
    return parser.parse_args()


In [ ]:
def main() -> None:
    args = parse_args()
    ensure_dirs()
    sns.set_theme(style="whitegrid", context="notebook")

    features_2025, features_2026 = load_features(args.top_n)
    train_2025 = model_frame(features_2025)
    keep_pitch_types = pd.Index(sorted(train_2025["pitch_type"].unique()))
    eval_2026 = model_frame(features_2026, keep_pitch_types) if not features_2026.empty else pd.DataFrame()
    cat, num, features = feature_columns(train_2025)

    X = train_2025[features]
    y = train_2025["pitch_type"]
    stratify = y if y.value_counts().min() >= 2 else None
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=stratify,
    )

    classes = np.asarray(sorted(y_train.unique()))
    metrics: list[dict[str, Any]] = []
    arsenal = build_pitcher_arsenal_table(train_2025.loc[X_train.index], y_train)
    arsenal_map = arsenal_map_from_table(arsenal)
    plot_pitcher_arsenals(arsenal)

    lookup_specs = {
        "lookup_global": ([], []),
        "lookup_pitcher": (["pitcher_name"], [[]]),
        "lookup_pitcher_count": (["pitcher_name", "count"], [["pitcher_name"], []]),
        "lookup_pitcher_prev": (["pitcher_name", "prev_pitch_type"], [["pitcher_name"], []]),
        "lookup_pitcher_count_prev": (
            ["pitcher_name", "count", "prev_pitch_type"],
            [["pitcher_name", "count"], ["pitcher_name", "prev_pitch_type"], ["pitcher_name"], []],
        ),
        "lookup_pitcher_count_prev_stand": (
            ["pitcher_name", "count", "prev_pitch_type", "stand"],
            [["pitcher_name", "count", "prev_pitch_type"], ["pitcher_name", "count"], ["pitcher_name"], []],
        ),
    }

    log("Evaluating empirical lookup baselines...")
    for name, (keys, fallbacks) in lookup_specs.items():
        if keys:
            probs = distribution_lookup_predict(X_train, y_train, X_test, classes, keys, fallbacks, alpha=1.0)
        else:
            counts = y_train.value_counts().reindex(classes, fill_value=0).astype(float)
            probs = np.tile((counts / counts.sum()).to_numpy(), (len(X_test), 1))
        add_unmasked_and_arsenal_metrics(
            metrics,
            name,
            "2025_holdout",
            classes,
            probs,
            y_test,
            X_test["pitcher_name"],
            arsenal_map,
        )
        if not eval_2026.empty:
            X_2026 = eval_2026[features]
            y_2026 = eval_2026["pitch_type"]
            if keys:
                probs_2026 = distribution_lookup_predict(X_train, y_train, X_2026, classes, keys, fallbacks, alpha=1.0)
            else:
                counts = y_train.value_counts().reindex(classes, fill_value=0).astype(float)
                probs_2026 = np.tile((counts / counts.sum()).to_numpy(), (len(X_2026), 1))
            add_unmasked_and_arsenal_metrics(
                metrics,
                name,
                "2026_to_date",
                classes,
                probs_2026,
                y_2026,
                X_2026["pitcher_name"],
                arsenal_map,
            )

    models = {
        "hist_gradient_boosting": make_hgb(cat, num),
        "random_forest": make_random_forest(cat, num),
    }

    fitted_models: dict[str, Pipeline] = {}
    log("Training model sweep...")
    for name, model in models.items():
        log(f"  training {name}")
        model.fit(X_train, y_train)
        fitted_models[name] = model
        probs = model_probs(model, X_test, classes)
        add_unmasked_and_arsenal_metrics(
            metrics,
            name,
            "2025_holdout",
            classes,
            probs,
            y_test,
            X_test["pitcher_name"],
            arsenal_map,
        )
        if not eval_2026.empty:
            X_2026 = eval_2026[features]
            y_2026 = eval_2026["pitch_type"]
            probs_2026 = model_probs(model, X_2026, classes)
            add_unmasked_and_arsenal_metrics(
                metrics,
                name,
                "2026_to_date",
                classes,
                probs_2026,
                y_2026,
                X_2026["pitcher_name"],
                arsenal_map,
            )

    metrics_df = pd.DataFrame(metrics).sort_values(["dataset", "accuracy"], ascending=[True, False])
    metrics_df.to_csv(OUTPUT_DIR / "next_pitch_model_sweep_metrics.csv", index=False)
    with open(OUTPUT_DIR / "next_pitch_model_sweep_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    entropy = context_entropy(X_train, y_train)
    plot_sweep(metrics_df)

    best_name = (
        metrics_df.loc[metrics_df["dataset"].eq("2025_holdout")]
        .sort_values(["accuracy", "top3_accuracy"], ascending=False)
        .iloc[0]["model"]
    )
    audit_base_name = "hist_gradient_boosting"
    if audit_base_name in fitted_models:
        best_model = fitted_models[audit_base_name]
        audit_name = f"{audit_base_name}_arsenal_masked"
        audits = [
            audit_best_model(
                audit_name,
                best_model,
                classes,
                X_test,
                y_test,
                train_2025,
                "2025_holdout",
                arsenal_map=arsenal_map,
            )
        ]
        if not eval_2026.empty:
            audits.append(
                audit_best_model(
                    audit_name,
                    best_model,
                    classes,
                    eval_2026[features],
                    eval_2026["pitch_type"],
                    eval_2026,
                    "2026_to_date",
                    arsenal_map=arsenal_map,
                )
            )
        audit = pd.concat(audits, ignore_index=True)
        audit.to_csv(OUTPUT_DIR / f"{audit_name}_prediction_audit.csv", index=False)
        plot_mix_for_best(audit, audit_name)

    log("\n=== Model sweep metrics ===")
    print(metrics_df.to_string(index=False), flush=True)

    log("\n=== Context uncertainty diagnostics ===")
    print(entropy.to_string(index=False), flush=True)

    log("\n=== Pitcher-specific arsenals ===")
    display_cols = ["pitcher_name", "pitch_type", "pitch_label", "pitches", "usage_rate", "arsenal_size"]
    print(
        arsenal.loc[arsenal["in_arsenal"], [c for c in display_cols if c in arsenal.columns]]
        .sort_values(["pitcher_name", "usage_rate"], ascending=[True, False])
        .to_string(index=False),
        flush=True,
    )

    log(f"\nBest 2025 holdout model by exact accuracy: {best_name}")


In [ ]:
if __name__ == "__main__":
    main()
